# Dietary Nitrate/Nitrite Exposure And Oral Microbiome

TRE-side exploratory notebook for testing whether diet-derived nitrate/nitrite/nitroso exposure groups differ in oral microbiome features.

This notebook follows the working `diet-mental_health.ipynb` pattern:

- participant data are loaded through `pheno_utils.PhenoLoader` inside TRE;
- diet exposures are built from the **full mega KG** by finding term-matching nodes and traversing back to `hpp_food:*` nodes;
- no 3 GB compound-reference scan is needed;
- intermediate analysis objects are cached as pickle files, not parquet, because some TRE kernels do not have `pyarrow` or `fastparquet`.

## Sections

1. Configuration and imports
2. Diet preprocessing into nitrate/nitrite exposure groups
3. Oral microbiome preprocessing
4. ADA filtering placeholder
5. Confounder loading and configuration
6. Statistical tests
7. Matplotlib plots
8. Summary tables

In [ ]:
# Imports and project setup
from __future__ import annotations

import json
import math
import os
import re
import sys
import time
import warnings
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# If root detection fails inside TRE, set this manually and rerun this cell, e.g.
# MANUAL_PROJECT_ROOT = Path("/home/ec2-user/studies/Diet_Data_Enhancement_Project")
MANUAL_PROJECT_ROOT = None


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    candidates = []
    env_root = os.environ.get("DIET_DATA_ENHANCEMENT_ROOT") or os.environ.get("PROJECT_ROOT")
    if env_root:
        candidates.append(Path(env_root).expanduser())
    if MANUAL_PROJECT_ROOT is not None:
        candidates.append(Path(MANUAL_PROJECT_ROOT).expanduser())
    candidates.extend([start, *start.parents])
    candidates.extend([
        Path.home() / "studies" / "Diet_Data_Enhancement_Project",
        Path.home() / "studies" / "Diet_Data_enhancement",
        Path.home() / "Diet_Data_Enhancement_Project",
        Path.home() / "Diet_Data_enhancement",
    ])
    seen = set()
    for candidate in candidates:
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        if (candidate / "outputs" / "visualizations" / "denovo_hpp_kg_mega_flexible_data.js").exists():
            return candidate
        if (candidate / "outputs" / "enhanced_hpp").exists() and (candidate / "downstream_analysis").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_DIR = PROJECT_ROOT / "downstream_analysis" / "manual" / "nitrate" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("OUTPUT_DIR =", OUTPUT_DIR)

In [ ]:
# Configuration
ID_COL = "participant_id"
FOOD_COL = "food_id"
REF_FOOD_COL = "hpp_food_id"
GRAMS_COL = "weight_g"

# Full mega KG, matching diet-mental_health.ipynb.
KG_MODE = "full_mega"
KG_MAX_HOPS = 4
MEGA_KG_DATA_PATH = PROJECT_ROOT / "outputs" / "visualizations" / "denovo_hpp_kg_mega_flexible_data.js"
FOOD_FEATURE_TABLE = PROJECT_ROOT / "outputs" / "enhanced_hpp" / "1.denovo" / "hpp_feature_matrix_per_100g.csv"

# Persistent caches. Pickle only; no parquet engine required.
FORCE_REBUILD_DIET_EXPOSURES = False
FORCE_REBUILD_ORAL_FEATURES = False
CACHE_DIR = OUTPUT_DIR / "cache_pickle"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_PATHS = {
    "diet_events": CACHE_DIR / "diet_events.pkl",
    "food_features": CACHE_DIR / "food_features.pkl",
    "kg_nodes": CACHE_DIR / "kg_nodes.pkl",
    "kg_edges": CACHE_DIR / "kg_edges.pkl",
    "kg_reverse_adj": CACHE_DIR / "kg_reverse_adj.pkl",
    "exposure_tables": CACHE_DIR / "exposure_tables.pkl",
    "exposure_summary": CACHE_DIR / "exposure_summary.pkl",
    "oral_features": CACHE_DIR / "oral_features.pkl",
    "confounders": CACHE_DIR / "confounders.pkl",
}

EXPOSURE_SPECS = {
    "vegetable_nitrate": {
        "terms": ["nitrate", "non metal nitrates"],
        "exclude_food_name_tokens": ["smoked", "cured", "sausage", "bacon", "ham", "salami"],
        "source_note": "Full mega KG path-connected nitrate foods, excluding obvious cured/smoked processed terms.",
    },
    "processed_nitrite": {
        "terms": ["nitrite", "sodium nitrite", "sodium nitrate", "non metal nitrites"],
        "include_food_name_tokens": ["smoked", "cured", "sausage", "bacon", "ham", "salami", "hot dog", "processed"],
        "source_note": "Full mega KG path-connected nitrite/preserved food signal.",
    },
    "nitroso_axis": {
        "terms": ["nitroso", "nitrosamine", "nitroso compound"],
        "source_note": "Exploratory full mega KG path-connected nitroso/nitrosamine signal.",
    },
    "arginine_no_axis": {
        "terms": ["arginine", "citrulline", "ornithine", "arginine and proline metabolism"],
        "source_note": "NO-biology support axis from full mega KG/nutrient/pathway terms.",
    },
}

GROUP_METHOD = "tertile"  # tertile, zero_vs_tertile, median
MIN_GROUP_N = 30
FDR_ALPHA = 0.10

GROUP_ORDER = ["low", "mid", "high"]
GROUP_COLORS = {
    "low": "#2166ac",
    "mid": "#f4a582",
    "high": "#b2182b",
    "bottom_10": "#2166ac",
    "top_10": "#b2182b",
}

# One-line confounder control. Add/remove names here, then rerun downward.
ACTIVE_CONFOUNDERS = ["age", "sex", "bmi", "smoking", "alcohol"]
# ACTIVE_CONFOUNDERS = ["age", "sex", "bmi", "smoking", "alcohol", "education", "physical_activity", "sleep_hours"]
# ACTIVE_CONFOUNDERS = []

PRIMARY_ORAL_OUTCOME_PATTERNS = [
    "raw_read_count", "shannon", "simpson", "richness", "alpha",
    "nitrate_reducer", "nitrogen", "nitrate", "nitrite", "nitros",
    "neisseria", "rothia", "veillonella", "actinomyces", "haemophilus", "prevotella", "kingella",
]
MAX_AUTO_ORAL_OUTCOMES = 40

## Helper Functions

In [ ]:
def flatten_index(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if isinstance(out.index, pd.MultiIndex) or out.index.name is not None:
        out = out.reset_index()
    return out.loc[:, ~out.columns.duplicated()].copy()


def normalize_ids(df: pd.DataFrame, id_col: str = ID_COL) -> pd.DataFrame:
    out = flatten_index(df)
    if id_col not in out.columns:
        aliases = ["research_stage_id", "user_id", "RegistrationCode", "participant", "sample_id"]
        for alias in aliases:
            if alias in out.columns:
                out = out.rename(columns={alias: id_col})
                break
    if id_col not in out.columns:
        raise ValueError(f"Could not find participant id column. Columns include: {out.columns.tolist()[:40]}")
    out[id_col] = out[id_col].astype(str)
    return out


def read_table(path: Path) -> pd.DataFrame:
    path = Path(path)
    if path.suffix.lower() in {".pkl", ".pickle"}:
        return pd.read_pickle(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path, low_memory=False)
    if path.suffix.lower() in {".tsv", ".txt"}:
        return pd.read_csv(path, sep="	", low_memory=False)
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported table: {path}")


def normalize_token_text(text: object) -> str:
    return re.sub(r"[^a-z0-9]+", " ", str(text).lower()).strip()


def token_contains(text: object, term: object) -> bool:
    clean_text = normalize_token_text(text)
    clean_term = normalize_token_text(term)
    if not clean_text or not clean_term:
        return False
    text_tokens = clean_text.split()
    term_tokens = clean_term.split()
    if len(term_tokens) == 1:
        return term_tokens[0] in text_tokens
    n = len(term_tokens)
    return any(text_tokens[i:i+n] == term_tokens for i in range(len(text_tokens)-n+1))


def contains_any(text: object, terms: list[str]) -> bool:
    return any(token_contains(text, term) for term in terms)


def finite_numeric(s: pd.Series) -> np.ndarray:
    return pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna().to_numpy(dtype=float)


def p_adjust_bh(p_values) -> np.ndarray:
    p = np.asarray(pd.to_numeric(pd.Series(p_values), errors="coerce"), dtype=float)
    q = np.full_like(p, np.nan, dtype=float)
    valid = np.isfinite(p)
    if valid.sum() == 0:
        return q
    pv = p[valid]
    order = np.argsort(pv)
    ranked = pv[order]
    n = len(ranked)
    adjusted = ranked * n / np.arange(1, n + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    adjusted = np.clip(adjusted, 0, 1)
    out = np.empty(n)
    out[order] = adjusted
    q[valid] = out
    return q


def cliffs_delta(x, y) -> float:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) == 0 or len(y) == 0:
        return np.nan
    ranks = stats.rankdata(np.concatenate([x, y]))
    rx = ranks[:len(x)].sum()
    u = rx - len(x) * (len(x) + 1) / 2
    return (2 * u / (len(x) * len(y))) - 1


def cohens_d(x, y) -> float:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) < 2 or len(y) < 2:
        return np.nan
    pooled = math.sqrt(((len(x)-1)*np.var(x, ddof=1) + (len(y)-1)*np.var(y, ddof=1)) / (len(x)+len(y)-2))
    if pooled == 0:
        return np.nan
    return (np.mean(y) - np.mean(x)) / pooled


def cohens_d_ci(d: float, n1: int, n2: int, z: float = 1.96) -> tuple[float, float]:
    if not np.isfinite(d) or n1 < 2 or n2 < 2:
        return np.nan, np.nan
    se = math.sqrt((n1 + n2) / (n1 * n2) + (d**2) / (2 * (n1 + n2 - 2)))
    return d - z * se, d + z * se

# 1. Diet Data Preprocessing Into Exposure Categories

Exposure discovery uses the **full mega KG**, not the compound reference CSVs. We find KG nodes matching nitrate/nitrite/nitroso terms, traverse reverse edges back to `hpp_food:*` nodes, then sum grams of those foods consumed by each participant.

In [ ]:
def load_diet_events_from_phenoloader() -> pd.DataFrame:
    """Load diet events using the same broad strategy as diet-mental_health.ipynb."""
    from pheno_utils import PhenoLoader
    from pheno_utils.config import DATASETS_PATH

    pl = PhenoLoader("diet_logging", age_sex_dataset=None, errors="warn")
    dataset_dir = Path(DATASETS_PATH) / pl.dataset
    candidate_paths = [dataset_dir / "diet_logging_events.parquet"]

    if "diet_logging_events" in getattr(pl, "dfs", {}):
        return normalize_ids(flatten_index(pl.dfs["diet_logging_events"]))

    if "diet_logging" in getattr(pl, "dfs", {}):
        summary_df = flatten_index(pl.dfs["diet_logging"])
        if "diet_logging_events" in summary_df.columns:
            for rel_path in summary_df["diet_logging_events"].dropna().astype(str).unique()[:20]:
                candidate_paths.append(dataset_dir / rel_path)

    events_path = next((path for path in candidate_paths if path.exists()), None)
    if events_path is None:
        PLACEHOLDER
    print("Reading diet events:", events_path, flush=True)
    return normalize_ids(pd.read_parquet(events_path))


def js_value_after_key(text: str, key: str):
    marker = f"{key}:"
    start = text.index(marker) + len(marker)
    decoder = json.JSONDecoder()
    value, _end = decoder.raw_decode(text[start:])
    return value


def load_mega_kg_data(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    if not path.exists():
        raise FileNotFoundError(f"Mega KG data file not found: {path}")
    print("Loading full mega KG:", path, flush=True)
    text = path.read_text(encoding="utf-8")
    kinds = js_value_after_key(text, "kinds")
    relations = js_value_after_key(text, "relations")
    compact_nodes = js_value_after_key(text, "nodes")
    compact_edges = js_value_after_key(text, "edges")

    nodes = pd.DataFrame(compact_nodes, columns=["key", "label", "kind_idx", "x", "y"])
    nodes["kind"] = nodes["kind_idx"].map(lambda i: kinds[int(i)])
    nodes = nodes[["key", "kind", "label"]]

    key_by_idx = nodes["key"].astype(str).to_numpy()
    edges = pd.DataFrame(compact_edges, columns=["source_idx", "target_idx", "relation_idx"])
    edges["source"] = key_by_idx[edges["source_idx"].astype(int).to_numpy()]
    edges["target"] = key_by_idx[edges["target_idx"].astype(int).to_numpy()]
    edges["relation"] = edges["relation_idx"].map(lambda i: relations[int(i)])
    edges = edges[["source", "target", "relation"]]
    return nodes, edges


def build_reverse_adjacency(kg_edges: pd.DataFrame) -> dict[str, set[str]]:
    reverse_adj: dict[str, set[str]] = {}
    for source, target in kg_edges[["source", "target"]].dropna().astype(str).itertuples(index=False):
        reverse_adj.setdefault(target, set()).add(source)
    return reverse_adj


def kg_target_nodes_for_terms(kg_nodes: pd.DataFrame, terms: list[str]) -> set[str]:
    target_nodes = set()
    search_cols = [c for c in ["key", "label", "kind"] if c in kg_nodes.columns]
    for row in kg_nodes[search_cols].fillna("").astype(str).itertuples(index=False, name=None):
        row_text = " | ".join(row)
        if any(token_contains(row_text, term) for term in terms):
            target_nodes.add(str(row[0]))
    return target_nodes


def hpp_food_id_from_node(node: str) -> str | None:
    node = str(node)
    if node.startswith("hpp_food:"):
        return node.split(":", 1)[1]
    return None


def kg_path_connected_food_ids(kg_nodes, kg_edges, reverse_adj, terms, max_hops=KG_MAX_HOPS) -> tuple[set[str], dict]:
    target_nodes = kg_target_nodes_for_terms(kg_nodes, terms)
    frontier = set(target_nodes)
    visited = set(target_nodes)
    food_ids = set()
    for depth in range(max_hops + 1):
        for node in frontier:
            food_id = hpp_food_id_from_node(node)
            if food_id is not None:
                food_ids.add(food_id)
        if depth == max_hops:
            break
        next_frontier = set()
        for node in frontier:
            next_frontier.update(reverse_adj.get(node, set()))
        next_frontier -= visited
        visited.update(next_frontier)
        frontier = next_frontier
        if not frontier:
            break
    return food_ids, {
        "kg_mode": KG_MODE,
        "target_node_count": len(target_nodes),
        "visited_node_count": len(visited),
        "connected_food_count": len(food_ids),
        "max_hops": max_hops,
        "example_target_nodes": " | ".join(sorted(target_nodes)[:15]),
    }


def load_food_features() -> pd.DataFrame:
    food = read_table(FOOD_FEATURE_TABLE)
    food[REF_FOOD_COL] = food[REF_FOOD_COL].astype(str)
    return food


def filter_connected_foods(food_ids: set[str], food_features: pd.DataFrame, spec: dict) -> set[str]:
    if not food_ids:
        return set()
    ref = food_features[food_features[REF_FOOD_COL].astype(str).isin({str(x) for x in food_ids})].copy()
    text_cols = [c for c in ["hpp_food_name", "hpp_product_name", "hpp_short_description", "hpp_category", "canonical_name", "canonical_category"] if c in ref.columns]
    if not text_cols:
        return {str(x) for x in food_ids}
    name_text = ref[text_cols].fillna("").astype(str).agg(" | ".join, axis=1)
    include = spec.get("include_food_name_tokens") or []
    exclude = spec.get("exclude_food_name_tokens") or []
    keep = pd.Series(True, index=ref.index)
    if include:
        keep &= name_text.map(lambda v: contains_any(v, include))
    if exclude:
        keep &= ~name_text.map(lambda v: contains_any(v, exclude))
    return set(ref.loc[keep, REF_FOOD_COL].astype(str))


def participant_exposure_from_kg_connected_foods(diet_events: pd.DataFrame, connected_food_ids: set[str], exposure_name: str) -> pd.DataFrame:
    food_col = FOOD_COL if FOOD_COL in diet_events.columns else "hpp_food_id"
    grams_col = GRAMS_COL if GRAMS_COL in diet_events.columns else "grams_consumed"
    d = diet_events[[ID_COL, food_col, grams_col]].copy()
    d[food_col] = d[food_col].astype(str)
    d[grams_col] = pd.to_numeric(d[grams_col], errors="coerce").fillna(0)
    connected = {str(x) for x in connected_food_ids}
    d["is_connected_food"] = d[food_col].isin(connected)
    d["exposure_value"] = np.where(d["is_connected_food"], d[grams_col], 0.0)
    out = d.groupby(ID_COL, as_index=False).agg(
        exposure_value=("exposure_value", "sum"),
        connected_food_events=("is_connected_food", "sum"),
        food_events=(food_col, "count"),
        unique_foods=(food_col, "nunique"),
        total_logged_g=(grams_col, "sum"),
    )
    out["exposure_per_1000g_logged"] = np.where(out["total_logged_g"] > 0, out["exposure_value"] / out["total_logged_g"] * 1000.0, np.nan)
    out["log1p_exposure_value"] = np.log1p(out["exposure_value"])
    out["exposure"] = exposure_name
    return out


def assign_exposure_groups(values: pd.Series, method: str = GROUP_METHOD) -> pd.Series:
    v = pd.to_numeric(values, errors="coerce").fillna(0)
    if method == "zero_vs_tertile":
        out = pd.Series("zero", index=v.index, dtype="object")
        positive = v[v > 0]
        if positive.empty:
            return out
        q1, q2 = positive.quantile([1/3, 2/3]).to_numpy()
        out[(v > 0) & (v <= q1)] = "low"
        out[(v > q1) & (v <= q2)] = "mid"
        out[v > q2] = "high"
        return out
    if method == "median":
        med = v.median()
        return pd.Series(np.where(v <= med, "low", "high"), index=v.index)
    positive = v[v > 0]
    out = pd.Series("low", index=v.index, dtype="object")
    if positive.nunique() < 3:
        out[v > 0] = "high"
        return out
    q1, q2 = positive.quantile([1/3, 2/3]).to_numpy()
    out[(v > q1) & (v <= q2)] = "mid"
    out[v > q2] = "high"
    return out


def assign_decile_groups(values: pd.Series) -> pd.Series:
    v = pd.to_numeric(values, errors="coerce").fillna(0)
    lo, hi = v.quantile([0.10, 0.90]).to_numpy()
    out = pd.Series(np.nan, index=v.index, dtype="object")
    out[v <= lo] = "bottom_10"
    out[v >= hi] = "top_10"
    return out

In [ ]:
# Build or load diet-derived exposure tables.
start = time.time()
if (not FORCE_REBUILD_DIET_EXPOSURES) and CACHE_PATHS["exposure_tables"].exists() and CACHE_PATHS["exposure_summary"].exists():
    print("Loading cached diet exposures:", CACHE_PATHS["exposure_tables"], flush=True)
    exposure_tables = pd.read_pickle(CACHE_PATHS["exposure_tables"])
    exposure_summary = pd.read_pickle(CACHE_PATHS["exposure_summary"])
    diet_events = pd.read_pickle(CACHE_PATHS["diet_events"]) if CACHE_PATHS["diet_events"].exists() else None
    food_features = pd.read_pickle(CACHE_PATHS["food_features"]) if CACHE_PATHS["food_features"].exists() else None
else:
    print("[1/5] Loading diet events via PhenoLoader...", flush=True)
    diet_events = load_diet_events_from_phenoloader()
    diet_events.to_pickle(CACHE_PATHS["diet_events"])
    print("diet_events", diet_events.shape, f"elapsed={time.time()-start:.1f}s", flush=True)

    print("[2/5] Loading food features...", flush=True)
    food_features = load_food_features()
    food_features.to_pickle(CACHE_PATHS["food_features"])
    print("food_features", food_features.shape, f"elapsed={time.time()-start:.1f}s", flush=True)

    print("[3/5] Loading full mega KG...", flush=True)
    if CACHE_PATHS["kg_nodes"].exists() and CACHE_PATHS["kg_edges"].exists():
        kg_nodes = pd.read_pickle(CACHE_PATHS["kg_nodes"])
        kg_edges = pd.read_pickle(CACHE_PATHS["kg_edges"])
    else:
        kg_nodes, kg_edges = load_mega_kg_data(MEGA_KG_DATA_PATH)
        kg_nodes.to_pickle(CACHE_PATHS["kg_nodes"])
        kg_edges.to_pickle(CACHE_PATHS["kg_edges"])
    print("kg_nodes", kg_nodes.shape, "kg_edges", kg_edges.shape, f"elapsed={time.time()-start:.1f}s", flush=True)

    print("[4/5] Building reverse adjacency...", flush=True)
    if CACHE_PATHS["kg_reverse_adj"].exists():
        kg_reverse_adj = pd.read_pickle(CACHE_PATHS["kg_reverse_adj"])
    else:
        kg_reverse_adj = build_reverse_adjacency(kg_edges)
        pd.to_pickle(kg_reverse_adj, CACHE_PATHS["kg_reverse_adj"])
    print("reverse adjacency targets", len(kg_reverse_adj), f"elapsed={time.time()-start:.1f}s", flush=True)

    print("[5/5] Building exposure tables from full KG paths...", flush=True)
    exposure_tables = {}
    exposure_summaries = []
    for i, (exposure_name, spec) in enumerate(EXPOSURE_SPECS.items(), start=1):
        step = time.time()
        print(f"  exposure {i}/{len(EXPOSURE_SPECS)}: {exposure_name}", flush=True)
        raw_connected, kg_detail = kg_path_connected_food_ids(
            kg_nodes, kg_edges, kg_reverse_adj, spec.get("terms", [exposure_name]), max_hops=KG_MAX_HOPS
        )
        connected = filter_connected_foods(raw_connected, food_features, spec)
        exposure = participant_exposure_from_kg_connected_foods(diet_events, connected, exposure_name)
        exposure["exposure_group"] = assign_exposure_groups(exposure["exposure_value"], method=GROUP_METHOD)
        exposure["decile_group"] = assign_decile_groups(exposure["exposure_value"])
        exposure_tables[exposure_name] = exposure
        detail = {
            "exposure": exposure_name,
            "source": "full_mega_kg_path_connected_food_grams",
            "terms": ", ".join(spec.get("terms", [])),
            "raw_connected_food_count": len(raw_connected),
            "connected_food_count_after_name_filters": len(connected),
            "participants": int(exposure[ID_COL].nunique()),
            "nonzero_participants": int((exposure["exposure_value"] > 0).sum()),
            "group_counts": exposure["exposure_group"].value_counts(dropna=False).to_dict(),
            "decile_counts": exposure["decile_group"].value_counts(dropna=False).to_dict(),
            **kg_detail,
        }
        exposure_summaries.append(detail)
        print(f"    connected={len(connected):,}, participants={detail['participants']:,}, nonzero={detail['nonzero_participants']:,}, elapsed={time.time()-step:.1f}s", flush=True)

    exposure_summary = pd.DataFrame(exposure_summaries)
    exposure_summary.to_pickle(CACHE_PATHS["exposure_summary"])
    pd.to_pickle(exposure_tables, CACHE_PATHS["exposure_tables"])
    print(f"Done exposure build. elapsed={time.time()-start:.1f}s", flush=True)

exposure_summary

In [ ]:
# Inspect connected foods for a selected exposure.
SELECTED_EXPOSURE_TO_INSPECT = "vegetable_nitrate"
if food_features is None and CACHE_PATHS["food_features"].exists():
    food_features = pd.read_pickle(CACHE_PATHS["food_features"])
exp = exposure_tables[SELECTED_EXPOSURE_TO_INSPECT]
connected_ids = set(exp.loc[exp["exposure_value"] > 0, ID_COL].astype(str))
# This table shows KG-connected food definitions, not participant rows.
# Recompute connected foods for inspection if KG objects are in memory/cache.
kg_nodes = pd.read_pickle(CACHE_PATHS["kg_nodes"])
kg_edges = pd.read_pickle(CACHE_PATHS["kg_edges"])
kg_reverse_adj = pd.read_pickle(CACHE_PATHS["kg_reverse_adj"])
spec = EXPOSURE_SPECS[SELECTED_EXPOSURE_TO_INSPECT]
raw_connected, _ = kg_path_connected_food_ids(kg_nodes, kg_edges, kg_reverse_adj, spec.get("terms", []), max_hops=KG_MAX_HOPS)
connected = filter_connected_foods(raw_connected, food_features, spec)
cols = [c for c in [REF_FOOD_COL, "hpp_food_name", "hpp_product_name", "hpp_category", "canonical_name", "canonical_category"] if c in food_features.columns]
food_features[food_features[REF_FOOD_COL].astype(str).isin(connected)][cols].head(100)

# 2. Oral Microbiome Preprocessing

This loads oral microbiome through `PhenoLoader('oral_microbiome')`. It first uses the main table (`pl['oral_microbiome']`) for directly available numeric outcomes and then tries to load linked MetaPhlAn/HUMAnN bulk files if paths are exposed in that table. If bulk pathway/taxon files are not available in your TRE mount, the notebook still proceeds with main-table oral metrics such as read count and any numeric columns present.

In [ ]:
NITRATE_REDUCER_PATTERNS = {
    "actinomyces": r"actinomyces",
    "haemophilus": r"haemophilus",
    "kingella": r"kingella",
    "neisseria": r"neisseria",
    "prevotella": r"prevotella",
    "rothia": r"rothia",
    "veillonella": r"veillonella",
}
NITROGEN_PATHWAY_RE = re.compile(r"nitrate|nitrite|nitric|nitros|denitrification|nitrogen|\bnar[A-Z]?\b|\bnir[A-Z]?\b|\bnor[A-Z]?\b|\bnos[Z]?\b", re.I)


def pheno_main_table(dataset: str, table: str | None = None) -> pd.DataFrame:
    from pheno_utils import PhenoLoader
    pl = PhenoLoader(dataset, errors="warn")
    table = table or dataset
    if table in getattr(pl, "dfs", {}):
        return normalize_ids(flatten_index(pl.dfs[table]))
    try:
        return normalize_ids(flatten_index(pl[table]))
    except Exception as exc:
        raise KeyError(f"Could not load {dataset}/{table}. Available pl.dfs: {list(getattr(pl, 'dfs', {}).keys())}") from exc


def choose_participant_id_col(df: pd.DataFrame) -> str | None:
    for col in [ID_COL, "participant_id", "sample_id", "sample_name", "wgs_dna_code"]:
        if col in df.columns:
            return col
    return None


def alpha_diversity_from_matrix(matrix: pd.DataFrame, prefix: str) -> pd.DataFrame:
    features = [c for c in matrix.select_dtypes(include="number").columns if c != ID_COL]
    if not features:
        return matrix[[ID_COL]].copy()
    x = matrix[features].clip(lower=0).fillna(0).to_numpy(dtype=float)
    row_sums = x.sum(axis=1)
    with np.errstate(divide="ignore", invalid="ignore"):
        p = np.divide(x, row_sums[:, None], out=np.zeros_like(x), where=row_sums[:, None] > 0)
        logp = np.where(p > 0, np.log(p), 0)
    return pd.DataFrame({
        ID_COL: matrix[ID_COL].astype(str).to_numpy(),
        f"{prefix}_shannon": -(p * logp).sum(axis=1),
        f"{prefix}_simpson": 1 - (p ** 2).sum(axis=1),
        f"{prefix}_richness": (x > 0).sum(axis=1),
        f"{prefix}_total_abundance": row_sums,
    })


def nitrate_reducer_from_matrix(matrix: pd.DataFrame, prefix: str) -> pd.DataFrame:
    numeric_cols = [c for c in matrix.select_dtypes(include="number").columns if c != ID_COL]
    out = pd.DataFrame({ID_COL: matrix[ID_COL].astype(str)})
    matched_all = []
    for name, pattern in NITRATE_REDUCER_PATTERNS.items():
        matches = [c for c in numeric_cols if re.search(pattern, c, re.I)]
        out[f"{prefix}_nitrate_reducer_{name}"] = matrix[matches].sum(axis=1) if matches else 0.0
        matched_all.extend(matches)
    out[f"{prefix}_nitrate_reducer_total"] = matrix[sorted(set(matched_all))].sum(axis=1) if matched_all else 0.0
    out[f"{prefix}_nitrate_reducer_matched_feature_count"] = len(set(matched_all))
    return out


def load_optional_bulk_table_from_paths(primary: pd.DataFrame, path_tokens: list[str]) -> pd.DataFrame | None:
    path_cols = [c for c in primary.columns if any(token in c.lower() for token in path_tokens)]
    for col in path_cols:
        for raw in primary[col].dropna().astype(str).unique()[:20]:
            path = Path(raw)
            candidates = [path, PROJECT_ROOT / raw]
            for candidate in candidates:
                if not candidate.exists():
                    continue
                try:
                    if candidate.suffix.lower() == ".csv":
                        return flatten_index(pd.read_csv(candidate, low_memory=False))
                    if candidate.suffix.lower() in {".tsv", ".txt"}:
                        return flatten_index(pd.read_csv(candidate, sep="	", low_memory=False))
                    if candidate.suffix.lower() in {".pkl", ".pickle"}:
                        return flatten_index(pd.read_pickle(candidate))
                    if candidate.suffix.lower() == ".parquet":
                        return flatten_index(pd.read_parquet(candidate))
                except Exception as exc:
                    print(f"Could not read optional bulk table {candidate}: {exc}")
    return None


def normalize_wide_microbiome_table(df: pd.DataFrame, prefix: str) -> pd.DataFrame | None:
    df = flatten_index(df)
    id_col = choose_participant_id_col(df)
    if id_col is None:
        return None
    df = df.rename(columns={id_col: ID_COL})
    numeric_cols = [c for c in df.select_dtypes(include="number").columns if c != ID_COL]
    if not numeric_cols:
        return None
    out = df[[ID_COL] + numeric_cols].copy()
    out[ID_COL] = out[ID_COL].astype(str)
    rename = {c: f"{prefix}_{re.sub(r'[^A-Za-z0-9]+', '_', str(c)).strip('_').lower()}" for c in numeric_cols}
    out = out.rename(columns=rename)
    return out.groupby(ID_COL, as_index=False).mean(numeric_only=True)


def build_oral_features() -> pd.DataFrame:
    primary = pheno_main_table("oral_microbiome", "oral_microbiome")
    print("oral primary", primary.shape, flush=True)
    numeric_cols = [c for c in primary.select_dtypes(include="number").columns if c != ID_COL]
    keep_numeric = [c for c in numeric_cols if any(token_contains(c, pat) for pat in PRIMARY_ORAL_OUTCOME_PATTERNS)]
    if not keep_numeric and "raw_read_count" in primary.columns:
        keep_numeric = ["raw_read_count"]
    oral_parts = [primary[[ID_COL] + keep_numeric].copy()] if keep_numeric else [primary[[ID_COL]].copy()]

    optional_specs = [
        ("oral_metaphlan_genus", ["metaphlan", "genus"]),
        ("oral_metaphlan_species", ["metaphlan", "species"]),
        ("oral_humann_pathway", ["humann", "pathway"]),
    ]
    for prefix, tokens in optional_specs:
        bulk = load_optional_bulk_table_from_paths(primary, tokens)
        if bulk is None:
            print(f"optional bulk not found/readable for {prefix}; continuing", flush=True)
            continue
        wide = normalize_wide_microbiome_table(bulk, prefix)
        if wide is None:
            print(f"optional bulk found but could not normalize for {prefix}; continuing", flush=True)
            continue
        oral_parts.append(wide)
        if "metaphlan" in prefix:
            oral_parts.append(alpha_diversity_from_matrix(wide, prefix))
            oral_parts.append(nitrate_reducer_from_matrix(wide, prefix))
        if "humann" in prefix:
            nitrogen_cols = [c for c in wide.select_dtypes(include="number").columns if NITROGEN_PATHWAY_RE.search(c)]
            if nitrogen_cols:
                tmp = wide[[ID_COL] + nitrogen_cols].copy()
                tmp[f"{prefix}_nitrogen_pathway_total"] = tmp[nitrogen_cols].sum(axis=1)
                tmp[f"{prefix}_nitrogen_pathway_matched_feature_count"] = len(nitrogen_cols)
                oral_parts.append(tmp)

    oral = oral_parts[0]
    for part in oral_parts[1:]:
        oral = oral.merge(part, on=ID_COL, how="outer")
    oral = oral.loc[:, ~oral.columns.duplicated()].copy()
    return normalize_ids(oral)


if CACHE_PATHS["oral_features"].exists() and not FORCE_REBUILD_ORAL_FEATURES:
    oral_features = pd.read_pickle(CACHE_PATHS["oral_features"])
    print("Loaded cached oral_features", oral_features.shape)
else:
    oral_features = build_oral_features()
    oral_features.to_pickle(CACHE_PATHS["oral_features"])
    print("Built oral_features", oral_features.shape)

oral_features.head()

In [ ]:
def select_oral_outcomes(oral: pd.DataFrame) -> list[str]:
    numeric_cols = [c for c in oral.select_dtypes(include="number").columns if c != ID_COL]
    preferred = [c for c in numeric_cols if any(token_contains(c, pat) for pat in PRIMARY_ORAL_OUTCOME_PATTERNS)]
    preferred = [c for c in preferred if not c.endswith("matched_feature_count")]
    if not preferred:
        preferred = numeric_cols
    return preferred[:MAX_AUTO_ORAL_OUTCOMES]

ORAL_OUTCOME_COLS = select_oral_outcomes(oral_features)
print("Selected oral outcomes", len(ORAL_OUTCOME_COLS))
ORAL_OUTCOME_COLS[:60]

# 3. ADA Filtering Placeholder

No filtering is applied now. Later add eligibility/QC filters here: antibiotic use, oral sample QC, medication exclusions, temporal alignment, pregnancy, dental procedures, minimum diet logging days, etc.

In [ ]:
def apply_ada_filters(df: pd.DataFrame, *, label: str = "analysis") -> pd.DataFrame:
    print(f"ADA filtering placeholder for {label}: keeping {len(df):,} rows.")
    return df.copy()

# 4. Confounders

Edit `ACTIVE_CONFOUNDERS` in the configuration cell and rerun downward. These are loaded via `PhenoLoader('lifestyle_and_environment')` and `PhenoLoader('sociodemographics')` where available.

In [ ]:
CONFOUNDER_SPECS = {
    "age": [("lifestyle_and_environment", "age_sex", ["age"]), ("sociodemographics", "age_sex", ["age"]), ("oral_microbiome", "age_sex", ["age"])],
    "sex": [("lifestyle_and_environment", "age_sex", ["sex"]), ("sociodemographics", "age_sex", ["sex"]), ("oral_microbiome", "age_sex", ["sex"])],
    "bmi": [("lifestyle_and_environment", "lifestyle_and_environment", ["bmi", "body_mass_index"])],
    "smoking": [("lifestyle_and_environment", "lifestyle_and_environment", ["smoking_present_vs_past", "smoking_status", "smoking_current", "smoke"])],
    "alcohol": [("lifestyle_and_environment", "lifestyle_and_environment", ["alcohol_current_frequency", "alcohol_past_status", "alcohol", "drinking_frequency"])],
    "physical_activity": [("lifestyle_and_environment", "lifestyle_and_environment", ["activity_moderate_days_weekly", "activity_vigorous_days_weekly", "physical_activity", "exercise"])],
    "sleep_hours": [("lifestyle_and_environment", "lifestyle_and_environment", ["sleep_hours_daily", "sleep_hours", "sleep_duration"])],
    "education": [("sociodemographics", "initial_medical", ["education", "education_level"]), ("sociodemographics", "ukbb", ["education", "education_level"])],
}


def load_pheno_table(dataset: str, table: str) -> pd.DataFrame | None:
    try:
        from pheno_utils import PhenoLoader
        pl = PhenoLoader(dataset, errors="warn")
        if table in getattr(pl, "dfs", {}):
            return normalize_ids(flatten_index(pl.dfs[table]))
        return normalize_ids(flatten_index(pl[table]))
    except Exception as exc:
        print(f"Could not load {dataset}/{table}: {exc}")
        return None


def resolve_possible_columns(df: pd.DataFrame, possible_cols: list[str]) -> list[str]:
    exact = [c for c in possible_cols if c in df.columns]
    if exact:
        return exact
    lower_to_col = {str(c).lower(): c for c in df.columns}
    found = []
    for wanted in possible_cols:
        wanted_l = wanted.lower()
        for col_l, col in lower_to_col.items():
            if wanted_l == col_l or wanted_l in col_l:
                found.append(col)
                break
    return list(dict.fromkeys(found))


def collapse_participant_table(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    keep = [ID_COL] + [c for c in cols if c in df.columns]
    tmp = df[keep].copy()
    numeric = tmp.select_dtypes(include="number").columns.difference([ID_COL]).tolist()
    other = [c for c in tmp.columns if c not in [ID_COL] + numeric]
    parts = []
    if numeric:
        parts.append(tmp.groupby(ID_COL, as_index=False)[numeric].mean())
    for c in other:
        parts.append(tmp.groupby(ID_COL)[c].agg(lambda s: s.dropna().iloc[-1] if len(s.dropna()) else np.nan).reset_index())
    if not parts:
        return pd.DataFrame(columns=[ID_COL])
    out = parts[0]
    for part in parts[1:]:
        out = out.merge(part, on=ID_COL, how="outer")
    return out


def build_confounder_table(active: list[str]) -> pd.DataFrame:
    if CACHE_PATHS["confounders"].exists():
        cached = pd.read_pickle(CACHE_PATHS["confounders"])
        # Use cache only when columns satisfy requested active confounders reasonably.
        if all((c in cached.columns) or any(col.startswith(c + "__") for col in cached.columns) for c in active):
            print("Loaded cached confounders", cached.shape)
            return cached
    tables = []
    summary = []
    for alias in active:
        found = False
        for dataset, table, possible in CONFOUNDER_SPECS.get(alias, []):
            df = load_pheno_table(dataset, table)
            if df is None:
                continue
            present = resolve_possible_columns(df, possible)
            if not present:
                continue
            part = collapse_participant_table(df, present)
            rename = {present[0]: alias} if len(present) == 1 else {c: f"{alias}__{c}" for c in present}
            part = part.rename(columns=rename)
            tables.append(part)
            summary.append({"confounder": alias, "dataset": dataset, "table": table, "columns": present, "rows": len(part)})
            found = True
            break
        if not found:
            summary.append({"confounder": alias, "dataset": None, "table": None, "columns": [], "rows": 0})
    display(pd.DataFrame(summary))
    out = pd.DataFrame(columns=[ID_COL])
    for part in tables:
        out = part if out.empty else out.merge(part, on=ID_COL, how="outer")
    out = normalize_ids(out) if not out.empty else out
    out.to_pickle(CACHE_PATHS["confounders"])
    return out

confounders = build_confounder_table(ACTIVE_CONFOUNDERS)
confounders.head()

# 5. Build Analysis Tables

In [ ]:
def build_analysis_table(exposure_name: str) -> pd.DataFrame:
    exposure = exposure_tables[exposure_name].copy()
    merged = exposure.merge(oral_features[[ID_COL] + ORAL_OUTCOME_COLS], on=ID_COL, how="inner")
    if not confounders.empty:
        merged = merged.merge(confounders, on=ID_COL, how="left")
    return apply_ada_filters(merged, label=exposure_name)

analysis_tables = {name: build_analysis_table(name) for name in exposure_tables}
for name, df in analysis_tables.items():
    print(name, df.shape, df["exposure_group"].value_counts(dropna=False).to_dict(), df["decile_group"].value_counts(dropna=False).to_dict())

# 6. Statistical Tests

In [ ]:
def design_matrix_for_confounders(df: pd.DataFrame, confounder_cols: list[str]) -> tuple[np.ndarray | None, pd.DataFrame]:
    present = [c for c in confounder_cols if c in df.columns]
    if not present:
        return None, pd.DataFrame(index=df.index)
    x = df[present].copy()
    for c in x.columns:
        if pd.api.types.is_numeric_dtype(x[c]):
            x[c] = pd.to_numeric(x[c], errors="coerce").fillna(pd.to_numeric(x[c], errors="coerce").median())
        else:
            x[c] = x[c].astype("category").cat.add_categories(["missing"]).fillna("missing")
    x = pd.get_dummies(x, drop_first=True, dtype=float)
    x = x.loc[:, x.nunique(dropna=False) > 1]
    if x.empty:
        return None, x
    x.insert(0, "intercept", 1.0)
    return x.to_numpy(dtype=float), x


def residualize(y: pd.Series, df: pd.DataFrame, confounder_cols: list[str]) -> np.ndarray:
    yy = pd.to_numeric(y, errors="coerce").to_numpy(dtype=float)
    X, _ = design_matrix_for_confounders(df, confounder_cols)
    if X is None:
        return yy
    ok = np.isfinite(yy) & np.isfinite(X).all(axis=1)
    resid = np.full_like(yy, np.nan, dtype=float)
    if ok.sum() <= X.shape[1] + 2:
        return yy
    beta = np.linalg.lstsq(X[ok], yy[ok], rcond=None)[0]
    resid[ok] = yy[ok] - X[ok].dot(beta)
    return resid


def low_high_tests(df, exposure_name, outcome, value_col=None):
    y_col = value_col or outcome
    low = finite_numeric(df.loc[df["exposure_group"] == "low", y_col])
    mid = finite_numeric(df.loc[df["exposure_group"] == "mid", y_col])
    high = finite_numeric(df.loc[df["exposure_group"] == "high", y_col])
    row = {"analysis": "tertile_low_mid_high", "exposure": exposure_name, "outcome": outcome, "n_low": len(low), "n_mid": len(mid), "n_high": len(high)}
    groups = [g for g in [low, mid, high] if len(g) >= MIN_GROUP_N]
    row["kruskal_p"] = stats.kruskal(*groups).pvalue if len(groups) >= 2 else np.nan
    if len(low) >= MIN_GROUP_N and len(high) >= MIN_GROUP_N:
        p = stats.mannwhitneyu(low, high, alternative="two-sided").pvalue
        d = cohens_d(low, high)
        ci_l, ci_h = cohens_d_ci(d, len(low), len(high))
        row.update({"low_vs_high_p": p, "cohens_d_high_minus_low": d, "cohens_d_ci_low": ci_l, "cohens_d_ci_high": ci_h, "cliffs_delta_high_minus_low": cliffs_delta(low, high), "median_diff_high_minus_low": np.median(high) - np.median(low)})
    else:
        row.update({"low_vs_high_p": np.nan, "cohens_d_high_minus_low": np.nan, "cohens_d_ci_low": np.nan, "cohens_d_ci_high": np.nan, "cliffs_delta_high_minus_low": np.nan, "median_diff_high_minus_low": np.nan})
    return row


def decile_tests(df, exposure_name, outcome, value_col=None):
    y_col = value_col or outcome
    bottom = finite_numeric(df.loc[df["decile_group"] == "bottom_10", y_col])
    top = finite_numeric(df.loc[df["decile_group"] == "top_10", y_col])
    row = {"analysis": "bottom10_vs_top10", "exposure": exposure_name, "outcome": outcome, "n_bottom_10": len(bottom), "n_top_10": len(top)}
    if len(bottom) >= MIN_GROUP_N and len(top) >= MIN_GROUP_N:
        p = stats.mannwhitneyu(bottom, top, alternative="two-sided").pvalue
        d = cohens_d(bottom, top)
        ci_l, ci_h = cohens_d_ci(d, len(bottom), len(top))
        row.update({"bottom_vs_top_p": p, "cohens_d_top_minus_bottom": d, "cohens_d_ci_low": ci_l, "cohens_d_ci_high": ci_h, "cliffs_delta_top_minus_bottom": cliffs_delta(bottom, top), "median_diff_top_minus_bottom": np.median(top) - np.median(bottom)})
    else:
        row.update({"bottom_vs_top_p": np.nan, "cohens_d_top_minus_bottom": np.nan, "cohens_d_ci_low": np.nan, "cohens_d_ci_high": np.nan, "cliffs_delta_top_minus_bottom": np.nan, "median_diff_top_minus_bottom": np.nan})
    return row


def ols_regression(df, exposure_name, outcome, confounder_cols):
    cols = ["log1p_exposure_value", outcome] + [c for c in confounder_cols if c in df.columns]
    sub = df[cols].copy().dropna(subset=["log1p_exposure_value", outcome])
    row = {"analysis": "continuous_ols", "exposure": exposure_name, "outcome": outcome, "n": len(sub)}
    if len(sub) < MIN_GROUP_N or sub["log1p_exposure_value"].nunique() < 3:
        row.update({"beta": np.nan, "standardized_beta": np.nan, "p_value": np.nan, "r2": np.nan})
        return row
    X_conf, _ = design_matrix_for_confounders(sub, confounder_cols)
    x = pd.to_numeric(sub["log1p_exposure_value"], errors="coerce").to_numpy(dtype=float)
    y = pd.to_numeric(sub[outcome], errors="coerce").to_numpy(dtype=float)
    X = np.column_stack([np.ones(len(sub)), x]) if X_conf is None else np.column_stack([X_conf, x])
    idx = X.shape[1] - 1
    ok = np.isfinite(y) & np.isfinite(X).all(axis=1)
    X, y = X[ok], y[ok]
    if len(y) <= X.shape[1] + 2:
        row.update({"beta": np.nan, "standardized_beta": np.nan, "p_value": np.nan, "r2": np.nan})
        return row
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    pred = X.dot(beta)
    resid = y - pred
    dof = len(y) - X.shape[1]
    sigma2 = (resid @ resid) / dof
    cov = sigma2 * np.linalg.pinv(X.T @ X)
    se = np.sqrt(np.diag(cov))
    t_stat = beta[idx] / se[idx] if se[idx] > 0 else np.nan
    p = 2 * stats.t.sf(abs(t_stat), dof) if np.isfinite(t_stat) else np.nan
    ss_tot = ((y - y.mean()) @ (y - y.mean()))
    r2 = 1 - ((resid @ resid) / ss_tot) if ss_tot > 0 else np.nan
    std_beta = beta[idx] * np.nanstd(x) / np.nanstd(y) if np.nanstd(y) > 0 else np.nan
    row.update({"beta": beta[idx], "standardized_beta": std_beta, "p_value": p, "r2": r2})
    return row

confounder_cols = [c for c in confounders.columns if c != ID_COL]
tertile_rows, decile_rows, reg_rows = [], [], []
tertile_adj_rows, decile_adj_rows = [], []
for exposure_name, df in analysis_tables.items():
    for outcome in ORAL_OUTCOME_COLS:
        tertile_rows.append(low_high_tests(df, exposure_name, outcome))
        decile_rows.append(decile_tests(df, exposure_name, outcome))
        reg_rows.append(ols_regression(df, exposure_name, outcome, confounder_cols))
        adj = df.copy()
        adj[f"{outcome}__resid"] = residualize(adj[outcome], adj, confounder_cols)
        r = low_high_tests(adj, exposure_name, outcome, value_col=f"{outcome}__resid")
        r["analysis"] = "tertile_low_mid_high_adjusted_residual"
        tertile_adj_rows.append(r)
        r = decile_tests(adj, exposure_name, outcome, value_col=f"{outcome}__resid")
        r["analysis"] = "bottom10_vs_top10_adjusted_residual"
        decile_adj_rows.append(r)

tertile_results = pd.DataFrame(tertile_rows + tertile_adj_rows)
decile_results = pd.DataFrame(decile_rows + decile_adj_rows)
regression_results = pd.DataFrame(reg_rows)
tertile_results["q_value"] = p_adjust_bh(tertile_results["low_vs_high_p"])
decile_results["q_value"] = p_adjust_bh(decile_results["bottom_vs_top_p"])
regression_results["q_value"] = p_adjust_bh(regression_results["p_value"])
tertile_results.sort_values("low_vs_high_p").head(30)

# 7. Matplotlib Publication-Style Plots

White background, black axes/grids, consistent exposure-group colors, and p-values printed on plots. Save lines are commented out.

In [ ]:
def format_p(p):
    if p is None or not np.isfinite(p):
        return "p=NA"
    return f"p={p:.1e}" if p < 1e-4 else f"p={p:.4f}"


def setup_axis(ax, title, x_title, y_title):
    ax.set_title(title, fontsize=13, color="black", pad=12)
    ax.set_xlabel(x_title, fontsize=12, color="black")
    ax.set_ylabel(y_title, fontsize=12, color="black")
    ax.set_facecolor("white")
    ax.figure.patch.set_facecolor("white")
    ax.grid(True, color="black", alpha=0.25, linewidth=0.7)
    for spine in ax.spines.values():
        spine.set_color("black")
    ax.tick_params(axis="both", colors="black", labelsize=11)


def p_value_for_plot(df, outcome, group_col, a, b):
    xa = finite_numeric(df.loc[df[group_col] == a, outcome])
    xb = finite_numeric(df.loc[df[group_col] == b, outcome])
    if len(xa) < MIN_GROUP_N or len(xb) < MIN_GROUP_N:
        return np.nan
    return stats.mannwhitneyu(xa, xb, alternative="two-sided").pvalue


def plot_group_distribution(df, exposure_name, outcome, group_col="exposure_group", kind="violin"):
    if group_col == "exposure_group":
        order, a, b = GROUP_ORDER, "low", "high"
        suffix = "low/mid/high exposure groups"
    else:
        order, a, b = ["bottom_10", "top_10"], "bottom_10", "top_10"
        suffix = "bottom/top 10% exposure groups"
    plot_df = df[df[group_col].isin(order)].copy()
    plot_df[outcome] = pd.to_numeric(plot_df[outcome], errors="coerce")
    values = [plot_df.loc[plot_df[group_col] == g, outcome].dropna().to_numpy(dtype=float) for g in order]
    p = p_value_for_plot(plot_df, outcome, group_col, a, b)
    fig, ax = plt.subplots(figsize=(7.5, 5.3), dpi=140)
    pos = np.arange(1, len(order) + 1)
    if kind == "violin":
        parts = ax.violinplot(values, positions=pos, showmedians=True, showextrema=False)
        for body, group in zip(parts["bodies"], order):
            body.set_facecolor(GROUP_COLORS[group])
            body.set_edgecolor("black")
            body.set_alpha(0.65)
        parts["cmedians"].set_color("black")
    else:
        box = ax.boxplot(values, positions=pos, patch_artist=True, widths=0.55, showfliers=False)
        for patch, group in zip(box["boxes"], order):
            patch.set_facecolor(GROUP_COLORS[group])
            patch.set_edgecolor("black")
            patch.set_alpha(0.65)
        for element in ["whiskers", "caps", "medians"]:
            for item in box[element]:
                item.set_color("black")
    rng = np.random.default_rng(123)
    means, sds, ns = [], [], []
    for ppos, vals in zip(pos, values):
        ns.append(len(vals))
        means.append(np.mean(vals) if len(vals) else np.nan)
        sds.append(np.std(vals, ddof=1) if len(vals) > 1 else np.nan)
        if len(vals):
            ax.scatter(np.full(len(vals), ppos) + rng.normal(0, 0.045, len(vals)), vals, s=13, color="black", alpha=0.35, linewidths=0)
    ax.errorbar(pos, means, yerr=sds, fmt="D", color="black", ecolor="black", capsize=7, markersize=5, label="mean +/- SD")
    ax.set_xticks(pos)
    ax.set_xticklabels([f"{g}\nn={n}" for g, n in zip(order, ns)])
    setup_axis(ax, f"{outcome}\n{exposure_name}: {suffix}", "Exposure group", outcome)
    ax.text(0.5, 0.98, f"{a} vs {b}: {format_p(p)}", transform=ax.transAxes, ha="center", va="top", bbox=dict(facecolor="white", edgecolor="black", boxstyle="square,pad=0.25"))
    ax.legend(frameon=False)
    fig.tight_layout()
    return fig, ax


def plot_continuous_regression(df, exposure_name, outcome):
    plot_df = df[["log1p_exposure_value", outcome]].copy().replace([np.inf, -np.inf], np.nan).dropna()
    fig, ax = plt.subplots(figsize=(7.5, 5.3), dpi=140)
    ax.scatter(plot_df["log1p_exposure_value"], plot_df[outcome], s=18, color="#4d4d4d", alpha=0.45, linewidths=0)
    if len(plot_df) >= MIN_GROUP_N and plot_df["log1p_exposure_value"].nunique() > 2:
        slope, intercept, r, p_raw, se = stats.linregress(plot_df["log1p_exposure_value"], plot_df[outcome])
        xs = np.linspace(plot_df["log1p_exposure_value"].min(), plot_df["log1p_exposure_value"].max(), 100)
        ax.plot(xs, intercept + slope * xs, color="#b2182b", linewidth=2.5)
    else:
        p_raw = np.nan
    sub = regression_results[(regression_results["exposure"] == exposure_name) & (regression_results["outcome"] == outcome)]
    txt = f"raw {format_p(p_raw)}"
    if not sub.empty:
        txt += f"\nadjusted {format_p(float(sub.iloc[0]['p_value']))}\nbeta={float(sub.iloc[0]['beta']):.3g}"
    ax.text(0.03, 0.97, txt, transform=ax.transAxes, ha="left", va="top", bbox=dict(facecolor="white", edgecolor="black", boxstyle="square,pad=0.25"))
    setup_axis(ax, f"Continuous exposure regression\n{exposure_name} vs {outcome}", "log1p exposure grams", outcome)
    fig.tight_layout()
    return fig, ax


def plot_effect_size_forest(results, analysis, effect_col, title, top_n=25):
    sub = results[results["analysis"] == analysis].copy()
    sub[effect_col] = pd.to_numeric(sub[effect_col], errors="coerce")
    sub = sub[np.isfinite(sub[effect_col])]
    pcol = "low_vs_high_p" if "low_vs_high_p" in sub.columns else "bottom_vs_top_p"
    sub = sub.sort_values(pcol).head(top_n).iloc[::-1].reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(9, max(5.5, 0.32 * len(sub) + 1.8)), dpi=140)
    y = np.arange(len(sub))
    labels = sub["exposure"] + " | " + sub["outcome"]
    xerr = np.vstack([sub[effect_col] - sub["cohens_d_ci_low"], sub["cohens_d_ci_high"] - sub[effect_col]])
    ax.errorbar(sub[effect_col], y, xerr=xerr, fmt="o", color="#2166ac", ecolor="black", capsize=4)
    ax.axvline(0, color="black", linestyle="--", linewidth=1)
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=9)
    setup_axis(ax, title, "Cohen's d", "Outcome")
    for yi, (_, row) in enumerate(sub.iterrows()):
        ax.text(row[effect_col], yi + 0.18, format_p(row[pcol]), fontsize=8, ha="center")
    fig.tight_layout()
    return fig, ax

In [ ]:
SELECTED_EXPOSURE = "vegetable_nitrate"
SELECTED_OUTCOME = ORAL_OUTCOME_COLS[0] if ORAL_OUTCOME_COLS else None
print("Selected", SELECTED_EXPOSURE, SELECTED_OUTCOME)
if SELECTED_OUTCOME:
    fig, ax = plot_group_distribution(analysis_tables[SELECTED_EXPOSURE], SELECTED_EXPOSURE, SELECTED_OUTCOME, "exposure_group", "violin")
    plt.show()
    # fig.savefig(OUTPUT_DIR / f"violin_{SELECTED_EXPOSURE}_{SELECTED_OUTCOME}.png", dpi=300, bbox_inches="tight")
    fig, ax = plot_group_distribution(analysis_tables[SELECTED_EXPOSURE], SELECTED_EXPOSURE, SELECTED_OUTCOME, "exposure_group", "box")
    plt.show()
    # fig.savefig(OUTPUT_DIR / f"box_{SELECTED_EXPOSURE}_{SELECTED_OUTCOME}.png", dpi=300, bbox_inches="tight")
    fig, ax = plot_group_distribution(analysis_tables[SELECTED_EXPOSURE], SELECTED_EXPOSURE, SELECTED_OUTCOME, "decile_group", "violin")
    plt.show()
    # fig.savefig(OUTPUT_DIR / f"violin_decile_{SELECTED_EXPOSURE}_{SELECTED_OUTCOME}.png", dpi=300, bbox_inches="tight")
    fig, ax = plot_continuous_regression(analysis_tables[SELECTED_EXPOSURE], SELECTED_EXPOSURE, SELECTED_OUTCOME)
    plt.show()
    # fig.savefig(OUTPUT_DIR / f"continuous_{SELECTED_EXPOSURE}_{SELECTED_OUTCOME}.png", dpi=300, bbox_inches="tight")

In [ ]:
fig, ax = plot_effect_size_forest(tertile_results, "tertile_low_mid_high", "cohens_d_high_minus_low", "Effect sizes: high vs low exposure", top_n=25)
plt.show()
# fig.savefig(OUTPUT_DIR / "effect_size_forest_tertile_high_vs_low.png", dpi=300, bbox_inches="tight")

fig, ax = plot_effect_size_forest(decile_results, "bottom10_vs_top10", "cohens_d_top_minus_bottom", "Effect sizes: top 10% vs bottom 10% exposure", top_n=25)
plt.show()
# fig.savefig(OUTPUT_DIR / "effect_size_forest_decile_top_vs_bottom.png", dpi=300, bbox_inches="tight")

# 8. Summary Tables

In [ ]:
tertile_summary = tertile_results.sort_values(["q_value", "low_vs_high_p"], na_position="last")
decile_summary = decile_results.sort_values(["q_value", "bottom_vs_top_p"], na_position="last")
regression_summary = regression_results.sort_values(["q_value", "p_value"], na_position="last")
tertile_summary.head(50)

In [ ]:
decile_summary.head(50)

In [ ]:
regression_summary.head(50)

In [ ]:
compact_tertile = tertile_summary.rename(columns={"low_vs_high_p": "primary_p", "cohens_d_high_minus_low": "effect_size", "median_diff_high_minus_low": "median_difference"}).copy()
compact_tertile["test_family"] = "low_mid_high"
compact_decile = decile_summary.rename(columns={"bottom_vs_top_p": "primary_p", "cohens_d_top_minus_bottom": "effect_size", "median_diff_top_minus_bottom": "median_difference"}).copy()
compact_decile["test_family"] = "bottom_top_10"
compact_reg = regression_summary.rename(columns={"p_value": "primary_p", "standardized_beta": "effect_size"}).copy()
compact_reg["test_family"] = "continuous_regression"
compact_reg["median_difference"] = np.nan
common = ["test_family", "analysis", "exposure", "outcome", "primary_p", "q_value", "effect_size", "median_difference"]
final_summary_table = pd.concat([
    compact_tertile[[c for c in common if c in compact_tertile.columns]],
    compact_decile[[c for c in common if c in compact_decile.columns]],
    compact_reg[[c for c in common if c in compact_reg.columns]],
], ignore_index=True, sort=False).sort_values(["q_value", "primary_p"], na_position="last")
final_summary_table.head(100)

In [ ]:
# Optional table export after finalizing settings.
# tertile_summary.to_csv(OUTPUT_DIR / "nitrate_oral_tertile_results.csv", index=False)
# decile_summary.to_csv(OUTPUT_DIR / "nitrate_oral_decile_results.csv", index=False)
# regression_summary.to_csv(OUTPUT_DIR / "nitrate_oral_continuous_regression_results.csv", index=False)
# final_summary_table.to_csv(OUTPUT_DIR / "nitrate_oral_final_summary_table.csv", index=False)